# 23 — The representativeness-block curation (study plan v0.17/.17.1, rule R0) and the manifest v3 freeze (zero solves)

**Trigger (R10.18 / AB R7.9–R7.11):** the 40-class EFG block rewarded cartographic artifacts of the GET indicative maps —
per-cell class value ∝ 1/footprint, so point records, envelope overshoots and clip slivers were the cheapest
"representativeness" in every solve. **Rule R0 (pre-screen, ahead of R1–R4):** every input must pass (i) a purpose-relevance
test, (ii) a map-validity test, and (iii) a boundary-proximity check for small retained classes. Curation ACCEPTED in
full (v0.17.1): anthropogenic biomes out (rule C, 12 classes); the point-record class out (rule A); sub-grain classes out
(rule B: footprint below one native GET cell, ~330 km²); F2.9 out (an envelope over 40% of the extent — uninformative);
TF1.6+TF1.7 and S1.1+SF1.1 merged (duplicate footprints) → **22 classes / 20 features**.

This notebook (kernel `y2y-geo`, minutes): (1) the curation table, (2) the curated block for BOTH stacks
(`input_data/aligned_stack/iucn_efg_v3/`, `aligned_stack_ab/iucn_efg_v3/`; v1 folders untouched), (3) the block card
v3 (rare-attainable count, the ≤1%-footprint companion, the southern statistic (E17 T2), the boundary-proximity
check (R0 iii), the rarity-scaled targets), (4) **manifest v3** — the 12 design formulations, weights re-derived
(EFGs sit outside the block accounting, so they equal v1's) with the EFG targets added — `spec/manifest_v3.csv` + hash.
v1/v2 manifests and `runs/` are superseded, never deleted. Then Ethan runs 12 → 18 (the re-solve) with `VERSION = "v3"`.

**v3.1 (study plan v0.17.3, 2026-09-09):** the boundary-proximity check flagged five of the six small retained classes as clip-edges; the ruling keeps all 22 classes and derives the rarity-scaled targets from each class's footprint in the study extent buffered by 250 km (100/500 km sensitivity) — the last cell. Manifest v3 (on-extent targets) is superseded by v3.1 before any solve.

In [1]:
# ---- bootstrap + the curation rules -------------------------------------------------------------------------
import importlib, json, hashlib, pathlib, shutil, sys
from datetime import datetime, timezone
import numpy as np
import pandas as pd
import rasterio
from scipy import ndimage
from pyproj import Transformer
_cands = [p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "config.py").exists()]
assert _cands, "config.py not found above the notebook"
ROOT = _cands[0]; sys.path.insert(0, str(ROOT))
import config, leverage_core as lc
for _m in (config, lc):
    importlib.reload(_m)
VP = config.y2y_paths()
assert VP.version in ("v3", "v3.1") and config.EFG_SUBDIR == VP.efg_subdir == "iucn_efg_v3", (VP.version, config.EFG_SUBDIR)
SPEC = ROOT / "analyses" / "y2y" / "spec"; REC = config.y2y_paths("v3").records; REC.mkdir(exist_ok=True)   # the v3 curation record
SRC, DST = config.HANDOFF_DIR / "iucn_efg", config.HANDOFF_DIR / config.EFG_SUBDIR
SRC_AB, DST_AB = config.AB_HANDOFF_DIR / "iucn_efg", config.AB_HANDOFF_DIR / config.EFG_SUBDIR
GRAIN_KM2 = 330            # rule B: one native GET grid cell (10 arcmin) at these latitudes
ENVELOPE_MIN_PCT = 1.0     # rule A: an envelope-method class below this % of the PU is a polygon edge, not an occurrence
BOUNDARY_KM, BOUNDARY_FLAG = 10, 0.50   # R0 (iii): share of a small retained class within 10 km of the study boundary
# the 40 classes on the parent extent: name, map-method category (GET map-details.xml), anthropogenic (rule C)
CLASSES = {
 "F2.6": ("Permanent salt and soda lakes", "mixed", False), "F2.2": ("Small permanent freshwater lakes", "proxy", False),
 "F1.4": ("Seasonal upland streams", "proxy", False), "F2.3": ("Seasonal freshwater lakes", "proxy", False),
 "F2.10": ("Subglacial lakes", "point", False), "F1.1": ("Permanent upland streams", "proxy", False),
 "T5.4": ("Cool deserts and semi-deserts", "proxy", False), "T6.1": ("Ice sheets, glaciers and perennial snowfields", "direct", False),
 "SF2.2": ("Flooded mines and other voids", "envelope", True), "F3.4": ("Freshwater aquafarms", "envelope", True),
 "F2.1": ("Large permanent freshwater lakes", "direct", False), "F1.6": ("Episodic arid rivers", "proxy", False),
 "T2.2": ("Deciduous temperate forests", "proxy", False), "T7.4": ("Urban and industrial ecosystems", "proxy", True),
 "F1.2": ("Permanent lowland rivers", "proxy", False), "T7.1": ("Annual croplands", "direct", True),
 "T4.4": ("Temperate woodlands", "envelope", False), "S2.1": ("Anthropogenic subterranean voids", "envelope", True),
 "T7.3": ("Plantations", "direct", True), "SF2.1": ("Water pipes and subterranean canals", "envelope", True),
 "T5.1": ("Semi-desert steppe", "proxy", False), "SF1.2": ("Groundwater ecosystems", "direct", False),
 "T7.2": ("Sown pastures and fields", "proxy", True), "TF1.6": ("Boreal, temperate and montane peat bogs", "envelope", False),
 "TF1.7": ("Boreal and temperate fens", "envelope", False), "T6.2": ("Polar/alpine cliffs, screes, outcrops and lava flows", "direct", False),
 "F3.1": ("Large reservoirs", "direct", True), "T7.5": ("Derived semi-natural pastures and old fields", "proxy", True),
 "T3.4": ("Young rocky pavements, lava flows and screes", "proxy", False), "T6.3": ("Polar tundra and deserts", "proxy", False),
 "TF1.2": ("Subtropical/temperate forested wetlands", "envelope", False), "F1.3": ("Freeze-thaw rivers and streams", "direct", False),
 "F3.5": ("Canals, ditches and drains", "envelope", True), "S1.1": ("Aerobic caves", "proxy", False),
 "SF1.1": ("Underground streams and pools", "proxy", False), "F3.2": ("Constructed lacustrine wetlands", "envelope", True),
 "F2.4": ("Freeze-thaw freshwater lakes", "direct", False), "F2.9": ("Geothermal pools and wetlands", "envelope", False),
 "T6.4": ("Temperate alpine grasslands and shrublands", "envelope", False), "T2.1": ("Boreal and temperate high montane forests and woodlands", "proxy", False)}
UTILITY_DROP = {"F2.9": "envelope over 40% of the extent: its target is met by any 30% of half the landscape (uninformative)"}
MERGES = {"TF1.6_TF1.7": ("TF1.6", "TF1.7"), "S1.1_SF1.1": ("S1.1", "SF1.1")}     # duplicate footprints -> one 1/n share
code_of = lambda p: p.stem.split(".web")[0]
files = {code_of(p): p for p in sorted(SRC.glob("*.tif"))}
assert set(files) == set(CLASSES), set(files) ^ set(CLASSES)

pu = lc.pu_mask(); n_pu = int(pu.sum())
with rasterio.open(config.HANDOFF_DIR / "mask_protected_areas.tif") as src:
    locked = (src.read(1) == 1) & pu
dist_in = ndimage.distance_transform_edt(pu)                     # km to the nearest non-PU cell, for cells inside the PU
rows = []
for code, p in files.items():
    name, method, anthro = CLASSES[code]
    e = np.nan_to_num(lc._read(p), nan=0.0) > 0
    pres = e & pu; n = int(pres.sum())
    r = dict(code=code, name=name, method=method, anthropogenic=anthro, cells=n, pct_pu=100 * n / n_pu,
             pct_in_PAs=100 * float((pres & locked).sum() / max(n, 1)),
             rule_C_purpose=anthro, rule_A_point=(method == "point"),
             rule_A_envelope_small=(method == "envelope" and 100 * n / n_pu < ENVELOPE_MIN_PCT),
             rule_B_subgrain=(method != "direct" and n < GRAIN_KM2), utility_drop=code in UTILITY_DROP,
             merged_into=next((m for m, pair in MERGES.items() if code in pair), ""))
    r["retained"] = not (r["rule_C_purpose"] or r["rule_A_point"] or r["rule_A_envelope_small"] or r["rule_B_subgrain"] or r["utility_drop"])
    r["boundary_share_10km"] = 100 * float(pres[dist_in <= BOUNDARY_KM].sum() / max(n, 1)) if (r["retained"] and r["pct_pu"] < 1.0) else np.nan
    r["clip_edge_flag"] = bool(r["boundary_share_10km"] > 100 * BOUNDARY_FLAG) if r["retained"] and r["pct_pu"] < 1.0 else False
    r["reason"] = ("; ".join([x for x, k in (("C purpose: anthropogenic biome", r["rule_C_purpose"]), ("A map-method: point-record class", r["rule_A_point"]),
                                            (f"A map-method: envelope < {ENVELOPE_MIN_PCT:g}% of the extent", r["rule_A_envelope_small"]),
                                            (f"B grain floor: < {GRAIN_KM2} km2 (one native GET cell)", r["rule_B_subgrain"]),
                                            (UTILITY_DROP.get(code, ""), r["utility_drop"])) if k]) if not r["retained"] else "")
    rows.append(r)
CUR = pd.DataFrame(rows).sort_values("cells").reset_index(drop=True)
RETAINED = sorted(CUR[CUR.retained].code)
EXPECT = {"F1.1", "T5.4", "T6.1", "F2.1", "F1.6", "T2.2", "F1.2", "T4.4", "T5.1", "SF1.2", "TF1.6", "TF1.7", "T6.2", "T3.4", "T6.3", "TF1.2", "F1.3", "S1.1", "SF1.1", "F2.4", "T6.4", "T2.1"}
assert set(RETAINED) == EXPECT, set(RETAINED) ^ EXPECT
FEATURES = [c for c in RETAINED if not CUR.set_index("code").loc[c, "merged_into"]] + list(MERGES)
assert len(RETAINED) == 22 and len(FEATURES) == 20
CUR.to_csv(REC / "efg_curation_v3.csv", index=False)
print(f"curation: {len(CUR)} classes -> {len(RETAINED)} retained -> {len(FEATURES)} features | dropped: "
      f"C {int(CUR.rule_C_purpose.sum())}, A-point {int(CUR.rule_A_point.sum())}, A-envelope<1% {int(CUR.rule_A_envelope_small.sum())} (both also C), "
      f"B-subgrain {int(CUR.rule_B_subgrain.sum())}, utility {int(CUR.utility_drop.sum())}")
print(CUR[["code", "name", "method", "cells", "pct_pu", "retained", "merged_into", "boundary_share_10km", "clip_edge_flag", "reason"]]
      .to_string(index=False, float_format=lambda v: f"{v:.2f}"))
print(f"\nR0 (iii) boundary-proximity flags (> {100*BOUNDARY_FLAG:.0f}% of the class within {BOUNDARY_KM} km of the study boundary): "
      f"{list(CUR[CUR.clip_edge_flag].code) or 'none'} -- reported, not auto-dropped")


curation: 40 classes -> 22 retained -> 20 features | dropped: C 12, A-point 1, A-envelope<1% 2 (both also C), B-subgrain 4, utility 1
 code                                                    name   method   cells  pct_pu  retained merged_into  boundary_share_10km  clip_edge_flag                                                                                              reason
 F2.6                           Permanent salt and soda lakes    mixed       3    0.00     False                              NaN           False                                                      B grain floor: < 330 km2 (one native GET cell)
 F2.2                        Small permanent freshwater lakes    proxy       9    0.00     False                              NaN           False                                                      B grain floor: < 330 km2 (one native GET cell)
 F1.4                                 Seasonal upland streams    proxy      93    0.01     False                              Na

In [2]:
# ---- build the curated block for BOTH stacks (v1 folders untouched) --------------------------------------------
def build_block(src, dst):
    dst.mkdir(exist_ok=True)
    have = {code_of(p): p for p in sorted(src.glob("*.tif"))}
    written, merged_into = [], set(m for pair in MERGES.values() for m in pair)
    for code in RETAINED:
        if code not in have or code in merged_into:
            continue
        out = dst / have[code].name
        if not out.exists():
            shutil.copy2(have[code], out)
        written.append(out)
    for mname, pair in MERGES.items():
        present = [have[c] for c in pair if c in have]
        if not present:
            continue
        out = dst / f"{mname}.web.merged_v3.tif"
        if not out.exists():
            with rasterio.open(present[0]) as s0:
                prof = s0.profile; a = s0.read(1)
            for q in present[1:]:
                with rasterio.open(q) as sq:
                    b = sq.read(1)
                nd = prof.get("nodata", 255)
                a = np.where((a == nd) & (b == nd), nd, np.maximum(np.where(a == nd, 0, a), np.where(b == nd, 0, b))).astype(a.dtype)
            with rasterio.open(out, "w", **prof) as d:
                d.write(a, 1)
        written.append(out)
    return sorted(written)
BUILT = build_block(SRC, DST); BUILT_AB = build_block(SRC_AB, DST_AB)
print(f"parent block: {len(BUILT)} features in {DST.relative_to(ROOT)} | Alberta block: {len(BUILT_AB)} features in {DST_AB.relative_to(ROOT)}")
assert len(BUILT) == 20, len(BUILT)
# verify: every written raster keeps the source profile; merged = cell-wise max of its pair
with rasterio.open(next(SRC.glob("*.tif"))) as s0:
    ref = {k: s0.profile[k] for k in ("dtype", "nodata", "width", "height", "crs", "transform")}
for out in BUILT:
    with rasterio.open(out) as d:
        assert {k: d.profile[k] for k in ref} == ref, out.name
for mname, pair in MERGES.items():
    with rasterio.open(DST / f"{mname}.web.merged_v3.tif") as d:
        m = d.read(1)
    parts = [rasterio.open(files[c]).read(1) for c in pair]
    assert np.array_equal(np.where(m == 255, 0, m), np.maximum(*[np.where(x == 255, 0, x) for x in parts])), mname
print("profiles identical to the sources; merged rasters = cell-wise max of their pairs")
importlib.reload(lc)
assert [p.stem for p in lc.efg_paths()] == sorted(p.stem for p in BUILT)
print(f"lc.efg_paths() now enumerates {len(lc.efg_paths())} features from config.EFG_SUBDIR = {config.EFG_SUBDIR!r}")


parent block: 20 features in input_data/aligned_stack/iucn_efg_v3 | Alberta block: 13 features in input_data/aligned_stack_ab/iucn_efg_v3
profiles identical to the sources; merged rasters = cell-wise max of their pairs
lc.efg_paths() now enumerates 20 features from config.EFG_SUBDIR = 'iucn_efg_v3'


In [3]:
# ---- block card v3: re-derived audit facts + the rarity-scaled targets -----------------------------------------
with rasterio.open(config.HANDOFF_DIR / "cost_uniform.tif") as src:
    tr = src.transform
rr, cc = np.where(pu)
lon, lat = Transformer.from_crs(config.TARGET_CRS, "EPSG:4326", always_xy=True).transform(tr.c + (cc + 0.5) * tr.a, tr.f + (rr + 0.5) * tr.e)
LAT = lat.astype(np.float32); south = LAT < 53.0
rows = []
for p in lc.efg_paths():
    v = np.nan_to_num(lc._read(p)[pu], nan=0.0); e = v > 0; n = int(e.sum())
    cmin, cmax, lev = lc.leverage_of(v)
    rows.append(dict(feature=p.stem, code=code_of(p), cells=n, pct_pu=100 * n / n_pu, pct_in_PAs=100 * float(e[locked[pu]].sum() / max(n, 1)),
                     cap_max=float(cmax), leverage=float(lev), rare_attainable=bool(cmax >= config.AUDIT["rare_cap"]),
                     le1pct_footprint=bool(n <= 0.01 * n_pu), south_share=float(e[south].sum() / max(n, 1)), mean_lat=float(LAT[e].mean()),
                     target=config.efg_target(n)))
CARD = pd.DataFrame(rows).sort_values("cells").reset_index(drop=True)
CARD.to_csv(REC / "efg_block_card_v3.csv", index=False)
n_rare, n_le1, n_south = int(CARD.rare_attainable.sum()), int(CARD.le1pct_footprint.sum()), int((CARD.south_share > 0.9).sum())
print(f"block card v3 ({len(CARD)} features): rare-attainable {n_rare}/{len(CARD)} (was 36/40) | <=1%-footprint companion {n_le1} (was 13) | "
      f">90% south of 53N {n_south}/{len(CARD)} (was 20/40) | median mean-latitude {CARD.mean_lat.median():.1f}N")
print(CARD[["code", "cells", "pct_pu", "pct_in_PAs", "cap_max", "rare_attainable", "le1pct_footprint", "south_share", "mean_lat", "target"]]
      .to_string(index=False, float_format=lambda v: f"{v:.3f}"))
TARGETS = {r.feature: round(float(r.target), 4) for r in CARD.itertuples()}
(REC / "efg_targets.json").write_text(json.dumps(dict(rule=config.EFG_TARGET_RULE, anchors=config.EFG_TARGET_ANCHORS,
    formula="t = 1 for footprint <= full_km2; floor_t for footprint >= floor_km2; log-linear in footprint between (Rodrigues et al. 2004)",
    targets=TARGETS), indent=1))
print(f"\nEFG targets ({config.EFG_TARGET_RULE}): " + ", ".join(f"{code_of(pathlib.Path(k))} {v:.2f}" for k, v in sorted(TARGETS.items(), key=lambda kv: kv[1])))


block card v3 (20 features): rare-attainable 17/20 (was 36/40) | <=1%-footprint companion 6 (was 13) | >90% south of 53N 9/20 (was 20/40) | median mean-latitude 52.6N
       code   cells  pct_pu  pct_in_PAs  cap_max  rare_attainable  le1pct_footprint  south_share  mean_lat  target
       F1.1    1429   0.112       0.700    1.000             True              True        1.000    46.165   0.942
       T5.4    2234   0.176       0.000    1.000             True              True        1.000    43.934   0.869
       T6.1    2730   0.214      48.645    1.000             True              True        0.725    51.778   0.836
       F2.1    6075   0.477      19.868    1.000             True              True        0.905    49.311   0.706
       F1.6    8215   0.645      19.270    1.000             True              True        1.000    42.652   0.657
       T2.2   11495   0.903       0.374    1.000             True              True        0.147    54.856   0.602
       F1.2   23187   1.822 

In [4]:
# ---- manifest v3: the 12 design formulations, weights re-derived (must equal v1), EFG targets added ----------------
AUDIT_OBJ = ROOT / "analyses" / "y2y" / "audit" / "audit_objects"
CONSTS = json.loads((AUDIT_OBJ / "audit_constants.json").read_text())
SC = json.loads((SPEC / "scenarios_v2.json").read_text())
V1 = pd.read_csv(SPEC / "manifest.csv").set_index("formulation_id")
sha_v1 = hashlib.sha256((SPEC / "manifest.csv").read_bytes()).hexdigest()
assert sha_v1 == (SPEC / "manifest_freeze.sha256").read_text().split()[0], "v1 manifest no longer matches its freeze hash -- stop"
sha_v2 = hashlib.sha256((SPEC / "manifest_v2.csv").read_bytes()).hexdigest() if (SPEC / "manifest_v2.csv").exists() else ""
REALIZATION = {"ssp585_2071_2100": None, "ssp245_2071_2100": config.REALIZATIONS_DIR / "macrorefugia_245_2071_2100.tif"}
sha245 = hashlib.sha256(REALIZATION["ssp245_2071_2100"].read_bytes()).hexdigest()
def _norm(d, what):
    tot = sum(d.values()); assert abs(tot - 1.0) < 5e-3, f"{what} sums to {tot:.6f}"
    return {k: v / tot for k, v in d.items()}
S0 = SC["S0_balanced"]; recipes = {}
for name in ("S0_balanced", "S1_core_habitat", "S2_connectivity", "S3_biodiversity", "S4_carbon"):
    s = SC[name]
    recipes[name.split("_")[0].lower()] = dict(scenario_name=name, shares=_norm(s["block_shares"], name), within={b: _norm(m, f"{name}/{b}") for b, m in s["within_block"].items()},
                                                targets=s["targets"], extra={}, regime="theta3_places" if name == "S4_carbon" else "theta5_amount")
recipes["s5"] = dict(scenario_name="S5_intactness", shares=_norm(S0["block_shares"], "S5"), within={b: _norm(m, f"S5/{b}") for b, m in S0["within_block"].items()},
                     targets=S0["targets"], extra={"human_modification": 10.0}, regime="theta5_amount")
efg_hashes = {p.stem: hashlib.sha256(p.read_bytes()).hexdigest() for p in lc.efg_paths()}
efg_block_sha = hashlib.sha256("\n".join(f"{k}:{v}" for k, v in sorted(efg_hashes.items())).encode()).hexdigest()
now = datetime.now(timezone.utc).isoformat(); rows = []
for climate in ("ssp585_2071_2100", "ssp245_2071_2100"):
    for sid in ("s0", "s1", "s2", "s3", "s4", "s5"):
        r = recipes[sid]
        lp = {"climate_type_macrorefugia": REALIZATION[climate]} if REALIZATION[climate] is not None else None
        d = lc.scenario_weights(r["shares"], within_block=r["within"], targets=r["targets"], layer_paths=lp)
        w = {x.feature: round(x.w, 6) for x in d.itertuples()}; w.update(r["extra"])
        prof = {x.feature: round(x.intended_share, 6) for x in d.itertuples()}
        fid = f"{sid}_{climate.split('_')[0]}_{r['regime'].split('_')[0]}"
        v1 = V1.loc[fid]; w1 = json.loads(v1.weight_vector)
        assert set(w) == set(w1) and all(abs(w[k] - w1[k]) < 1e-6 for k in w), f"{fid}: weights differ from v1 (EFGs are outside the block accounting)"
        hashes = dict(CONSTS["layer_sha256"]); hashes.update(efg_hashes)
        if REALIZATION[climate] is not None:
            hashes["climate_type_macrorefugia"] = sha245
        rows.append(dict(formulation_id=fid, scenario_id=sid, scenario_name=r["scenario_name"], climate_level=climate, carbon_regime=r["regime"],
                         budget_pct=config.BUDGET_PCT, weight_vector=json.dumps(w), target_vector=json.dumps({**r["targets"], **TARGETS}),
                         influence_profile_intended=json.dumps(prof), k_requested=50, band_gap_g=0.05, opt_gap=1e-4, numeric_focus=2,
                         dust_rule_version=v1.dust_rule_version, role="design", estimator=v1.estimator, verdict_rule=v1.verdict_rule, solver="gurobi",
                         seed_policy=v1.seed_policy, input_layer_hashes=json.dumps(hashes), kbest_ref="", twin_ref="",
                         manifest_version=3, efg_block_version="v3", efg_block_sha256=efg_block_sha, efg_features=json.dumps(sorted(efg_hashes)),
                         efg_target_rule=config.EFG_TARGET_RULE, supersedes=f"v1 {sha_v1[:16]} / v2 {sha_v2[:16]}",
                         trigger="study plan v0.17 (R10.18): EFG block curation under rule R0 -- 40 -> 22 classes / 20 features",
                         created_utc=now, frozen=True))
M3 = pd.DataFrame(rows); assert M3.formulation_id.is_unique and len(M3) == 12
V3 = config.y2y_paths("v3")
if V3.manifest.exists() and V3.freeze.exists() and hashlib.sha256(V3.manifest.read_bytes()).hexdigest() == V3.freeze.read_text().split()[0]:
    d3 = V3.freeze.read_text().split()[0]
    print(f"manifest v3 already frozen ({d3[:16]}...) -- kept byte-identical (supersede, never delete)")
else:
    M3.to_csv(V3.manifest, index=False)
    d3 = hashlib.sha256(V3.manifest.read_bytes()).hexdigest()
    V3.freeze.write_text(f"{d3}  {V3.manifest.name}\n")
    print(f"FROZEN: {V3.manifest.relative_to(ROOT)} (12 design formulations; EFG block v3 sha {efg_block_sha[:16]}...; targets {config.EFG_TARGET_RULE}) | sha256 {d3[:16]}...")
print("v3 (on-extent targets) is SUPERSEDED by v3.1 (window-derived targets, next cell) before any solve -- study plan v0.17.3")


manifest v3 already frozen (e47991bcdcd3c16e...) -- kept byte-identical (supersede, never delete)
v3 (on-extent targets) is SUPERSEDED by v3.1 (window-derived targets, next cell) before any solve -- study plan v0.17.3


In [5]:
# ---- v3.1 (study plan v0.17.3): targets from each class's footprint in the BUFFERED REGIONAL WINDOW ------------
# Rarity measured inside the study polygon made range edges of classes abundant just outside the line look rare
# (R0 iii flagged 5 of the 6 small classes). Zonal count on the GET archive maps within the extent buffered by 250 km
# (100 / 500 km as sensitivity): widespread-outside classes -> ~0.10 targets; classes rare in the window pin legitimately.
import rasterio.features as rfeatures
from rasterio.warp import reproject, Resampling
from rasterio.transform import from_origin
import geopandas as gpd
V31 = config.y2y_paths("v3.1"); REC31 = V31.records; REC31.mkdir(exist_ok=True)
SRC_GET = config.DATASETS["iucn_efg"]["path"]
W_MAIN, W_SENS = config.EFG_TARGET_WINDOW_KM, config.EFG_TARGET_WINDOW_SENS_KM
WINDOWS = sorted({W_MAIN, *W_SENS})
polys = {w: config.study_area(config.BUFFER_KM + w).union_all() for w in WINDOWS}      # extent (= boundary + 20 km) + w
big = polys[max(WINDOWS)]
x0, y0, x1, y1 = big.bounds; res = config.TARGET_RES_M
x0, y1 = np.floor(x0 / res) * res, np.ceil(y1 / res) * res
W_, H_ = int(np.ceil((x1 - x0) / res)), int(np.ceil((y1 - y0) / res))
tr_win = from_origin(x0, y1, res, res)
win_masks = {w: rfeatures.rasterize([(polys[w], 1)], out_shape=(H_, W_), transform=tr_win, fill=0, dtype="uint8").astype(bool) for w in WINDOWS}
ext_mask = rfeatures.rasterize([(config.study_area().union_all(), 1)], out_shape=(H_, W_), transform=tr_win, fill=0, dtype="uint8").astype(bool)
# lon/lat read window covering the big window (source: global WGS84 30-arcsecond, NoData 0, values 1 major / 2 minor)
from pyproj import Transformer
_tf = Transformer.from_crs(config.TARGET_CRS, "EPSG:4326", always_xy=True)
gx = np.linspace(x0, x0 + W_ * res, 60); gy = np.linspace(y1 - H_ * res, y1, 60)
lon, lat = _tf.transform(*np.meshgrid(gx, gy))
bb = (lon.min() - 1, lat.min() - 1, lon.max() + 1, lat.max() + 1)
def window_presence(code):
    with rasterio.open(SRC_GET / files[code].name) as src:
        rw = rasterio.windows.from_bounds(*bb, src.transform).round_offsets().round_lengths()
        arr = src.read(1, window=rw); src_tr = src.window_transform(rw); src_crs = src.crs
    out = np.zeros((H_, W_), np.uint8)
    reproject(arr, out, src_transform=src_tr, src_crs=src_crs, dst_transform=tr_win, dst_crs=config.TARGET_CRS,
              resampling=Resampling.nearest, src_nodata=0, dst_nodata=0)
    return out > 0
rows = []; PRES = {}
for code in RETAINED:
    PRES[code] = window_presence(code)
feat_of = {}
for code in RETAINED:
    m = CUR.set_index("code").loc[code, "merged_into"]
    feat_of.setdefault(m if m else code, []).append(code)
stem_of = {code_of(p): p.stem for p in lc.efg_paths()} | {m: next(p.stem for p in lc.efg_paths() if p.stem.startswith(m + ".")) for m in MERGES}
for feat, codes in feat_of.items():
    pres = np.zeros((H_, W_), bool)
    for c in codes:
        pres |= PRES[c]
    r = dict(feature=stem_of[feat], code=feat, extent_km2=int((pres & ext_mask).sum()))
    for w in WINDOWS:
        r[f"window{w}_km2"] = int((pres & win_masks[w]).sum())
        r[f"window{w}_pct"] = 100 * r[f"window{w}_km2"] / int(win_masks[w].sum())
        r[f"target_window{w}"] = config.efg_target(r[f"window{w}_km2"])
    r["share_of_window250_inside_extent"] = r["extent_km2"] / max(r[f"window{W_MAIN}_km2"], 1)
    r["rare_window"] = bool(r[f"window{W_MAIN}_km2"] <= config.EFG_RARE_WINDOW_PCT * int(win_masks[W_MAIN].sum()))
    for w in W_SENS:
        r[f"rare_window{w}"] = bool(r[f"window{w}_km2"] <= config.EFG_RARE_WINDOW_PCT * int(win_masks[w].sum()))
    r["target_on_extent_v3"] = TARGETS[stem_of[feat]]
    r["target_v31"] = r[f"target_window{W_MAIN}"]
    rows.append(r)
WF = pd.DataFrame(rows).sort_values(f"window{W_MAIN}_km2").reset_index(drop=True)
WF.to_csv(REC31 / "efg_window_footprints.csv", index=False)
print(f"window areas: " + ", ".join(f"+{w} km = {int(win_masks[w].sum()):,} km2" for w in WINDOWS) + f" | extent {int(ext_mask.sum()):,} km2")
print(WF[["code", "extent_km2", f"window{W_MAIN}_km2", f"window{W_MAIN}_pct", "share_of_window250_inside_extent", "rare_window", "target_on_extent_v3", "target_v31"] +
         [f"target_window{w}" for w in W_SENS]].to_string(index=False, float_format=lambda v: f"{v:.3f}"))
print(f"\nrare in the +{W_MAIN} km window (<= {100*config.EFG_RARE_WINDOW_PCT:g}% of it): {list(WF[WF.rare_window].code) or 'none'}"
      + "".join(f" | +{w} km: {list(WF[WF[f'rare_window{w}']].code) or 'none'}" for w in W_SENS))
T31 = dict(zip(WF.feature, WF.target_v31.round(4)))
(REC31 / "efg_targets.json").write_text(json.dumps(dict(rule="loglinear", window_km=W_MAIN, sensitivity_km=list(W_SENS), anchors=config.EFG_TARGET_ANCHORS,
    formula="t from the class footprint within the study extent buffered by window_km (zonal count on the GET archive maps, 1 km nearest warp): "
            "1 for <= full_km2, floor_t for >= floor_km2, log-linear between", targets=T31), indent=1))
# manifest v3.1 = v3 rows with the EFG part of every target vector replaced; weights unchanged; write once
M31 = pd.read_csv(V3.manifest)
for i, r in M31.iterrows():
    tv = json.loads(r.target_vector); tv = {k: v for k, v in tv.items() if k not in TARGETS}; tv.update(T31)
    M31.at[i, "target_vector"] = json.dumps(tv)
M31["manifest_version"] = 3.1; M31["efg_target_rule"] = f"loglinear_window{W_MAIN}km"
M31["supersedes"] = f"v3 {d3[:16]} (on-extent targets, never solved) / " + M31.supersedes.iloc[0]
M31["trigger"] = "study plan v0.17.3: clip-edge resolution -- rarity-scaled targets from the buffered regional window (R0 iii)"
M31["created_utc"] = datetime.now(timezone.utc).isoformat()
if V31.manifest.exists() and V31.freeze.exists() and hashlib.sha256(V31.manifest.read_bytes()).hexdigest() == V31.freeze.read_text().split()[0]:
    print(f"manifest v3.1 already frozen ({V31.freeze.read_text().split()[0][:16]}...) -- kept byte-identical")
else:
    M31.to_csv(V31.manifest, index=False)
    d31 = hashlib.sha256(V31.manifest.read_bytes()).hexdigest(); V31.freeze.write_text(f"{d31}  {V31.manifest.name}\n")
    print(f"FROZEN: {V31.manifest.relative_to(ROOT)} (12 design formulations; targets from the +{W_MAIN} km window) | sha256 {d31[:16]}...")
print("next: 12 (VERSION v3.1, ~10-12 h) -> 13 -> 15 -> 18 (v3.1, ~12 h) -> 18b -> 18c -> 19 -> 20 -- or one command: caffeinate -i bash analyses/y2y/run_v31.sh")


window areas: +100 km = 2,435,283 km2, +250 km = 3,813,704 km2, +500 km = 6,373,984 km2 | extent 1,551,653 km2
       code  extent_km2  window250_km2  window250_pct  share_of_window250_inside_extent  rare_window  target_on_extent_v3  target_v31  target_window100  target_window500
       F1.1        4228          21318          0.559                             0.198         True                0.942       0.501             0.625             0.287
       F2.1        8847          22674          0.595                             0.390         True                0.706       0.491             0.569             0.464
       T6.1       10714          51084          1.339                             0.210        False                0.836       0.359             0.472             0.230
       F1.2       35600         102648          2.692                             0.347        False                0.488       0.245             0.326             0.157
       T2.2       20813         186703 

## Next
Commit `spec/manifest_v3.csv` + `manifest_v3.sha256` + `spec/v3/` (the curation record). The Alberta re-run (stage 2)
uses `aligned_stack_ab/iucn_efg_v3/` built above, once its spec mirror is re-pinned.